In [3]:
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Load and clean the periodic-table dataset
# ---------------------------------------------------------

csv_path = "0 PERIODIC TABLE DATA CUSTOMIZED FOR OQMD.csv"

periodic_table = pd.read_csv(csv_path)

# The element symbols in the CSV contain line breaks, such as "\nCa\n".
periodic_table["name"] = periodic_table["name"].astype(str).str.strip()

# Keep the CSV's exact property order.
# gtf and of are excluded because their columns contain no elemental data.
property_columns = [
    column
    for column in periodic_table.columns
    if column not in ["name", "gtf", "of"]
]

# Convert all property columns to numeric values.
# Any blank strings become NaN.
periodic_table[property_columns] = periodic_table[property_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Make the element symbol the lookup index.
periodic_table = periodic_table.set_index("name")


# ---------------------------------------------------------
# 2. Parse a formula written in ABX3 form
# ---------------------------------------------------------

def parse_abx3_formula(formula):
    """
    Parse an ABX3 formula such as CaInBr3.

    Returns:
        A symbol, B symbol, X symbol
    """

    # Remove spaces, if any.
    formula = formula.replace(" ", "")

    # Match:
    # A = first element
    # B = second element
    # X = third element
    # followed by the number 3
    match = re.fullmatch(
        r"([A-Z][a-z]?)([A-Z][a-z]?)([A-Z][a-z]?)3",
        formula
    )

    if match is None:
        raise ValueError(
            f"{formula!r} is not in the expected ABX3 format. "
            "Example valid formulas: CaInBr3, KCuCl3, CsPbI3."
        )

    A, B, X = match.groups()

    for symbol in [A, B, X]:
        if symbol not in periodic_table.index:
            raise ValueError(
                f"Element {symbol!r} was not found in the periodic-table CSV."
            )

    return A, B, X


# ---------------------------------------------------------
# 3. Calculate weighted mean and standard deviation
# ---------------------------------------------------------

def calculate_abx3_features(formula):
    """
    Calculate weighted elemental-property features for an ABX3 compound.

    A is counted once.
    B is counted once.
    X is counted three times.

    Returns:
        feature_array:
            NumPy array ordered as:
            z_mean, z_std, grp_mean, grp_std, ...

        feature_names:
            Names corresponding to each value in feature_array
    """

    A, B, X = parse_abx3_formula(formula)

    # Stoichiometric weights for ABX3.
    weights = np.array([1.0, 1.0, 3.0])

    feature_values = []
    feature_names = []

    for property_name in property_columns:
        values = np.array(
            [
                periodic_table.loc[A, property_name],
                periodic_table.loc[B, property_name],
                periodic_table.loc[X, property_name]
            ],
            dtype=float
        )

        # The user said relevant formulas should not contain elements
        # with missing values, so raise a useful error if one appears.
        if np.isnan(values).any():
            missing_elements = [
                symbol
                for symbol, value in zip([A, B, X], values)
                if np.isnan(value)
            ]

            raise ValueError(
                f"Missing {property_name!r} data for: "
                f"{', '.join(missing_elements)}"
            )

        # Weighted mean:
        # (A + B + 3X) / 5
        weighted_mean = np.average(values, weights=weights)

        # Weighted population variance:
        # [1(A-mean)^2 + 1(B-mean)^2 + 3(X-mean)^2] / 5
        weighted_variance = np.average(
            (values - weighted_mean) ** 2,
            weights=weights
        )

        weighted_std = np.sqrt(weighted_variance)

        feature_values.extend([weighted_mean, weighted_std])
        feature_names.extend(
            [
                f"{property_name}_mean",
                f"{property_name}_std"
            ]
        )

    return np.array(feature_values, dtype=float), feature_names


# ---------------------------------------------------------
# 4. Example
# ---------------------------------------------------------

compound = "CaInBr3"

feature_array, feature_names = calculate_abx3_features(compound)

print("Compound:", compound)
print("\nFeature names:")
print(feature_names)

print("\nFeature array:")
print(feature_array)

print("\nNamed values:")
for name, value in zip(feature_names, feature_array):
    print(f"{name}: {value}")

Compound: CaInBr3

Feature names:
['z_mean', 'z_std', 'grp_mean', 'grp_std', 'row_mean', 'row_std', 'av_ionrad_mean', 'av_ionrad_std', 'atom_rad_mean', 'atom_rad_std', 'x_mean', 'x_std', 'val_mean', 'val_std', 'ea_mean', 'ea_std', 'ie_mean', 'ie_std', 'atom_mass_mean', 'atom_mass_std', 'bp_mean', 'bp_std', 'rho_mean', 'rho_std', 'mp_mean', 'mp_std', 'k_mean', 'k_std', 'heat_fus_mean', 'heat_fus_std', 'heat_vap_mean', 'heat_vap_std', 'spec_heat_mean', 'spec_heat_std']

Feature array:
[3.48000000e+01 9.17387595e+00 1.32000000e+01 5.81033562e+00
 4.20000000e+00 4.00000000e-01 9.45500000e-01 9.97672291e-02
 1.38200000e+00 2.87569122e-01 2.33200000e+00 8.07722725e-01
 5.20000000e+00 2.22710575e+00 2.08332000e+00 1.57092215e+00
 9.46820000e+00 2.87461978e+00 7.89216000e+01 2.36654690e+01
 1.01960000e+03 8.62418135e+02 3.64200000e+00 1.93339494e+00
 4.68042000e+02 3.28247306e+02 5.64720000e+01 7.84584084e+01
 6.04600000e-02 1.73068310e-02 8.89800000e-01 9.34666443e-01
 7.41180000e-01 2.820790

In [6]:
import itertools
import math
import pandas as pd
from pymatgen.core import Element, Species

# --------------------------------------------------
# Elements to consider
# --------------------------------------------------

A_ELEMENTS = [
    "Li","Na","K","Rb","Cs",
    "Mg","Ca","Sr","Ba",
    "Sc","Y","La","Ce","Pr","Nd","Sm","Eu","Gd",
    "Tb","Dy","Ho","Er","Tm","Yb","Lu",
    "Bi","Pb"
]

B_ELEMENTS = [
    "Al","Ga","In","Tl",
    "Si","Ge","Sn","Pb",
    "Sc","Y",
    "Ti","Zr","Hf",
    "V","Nb","Ta",
    "Cr","Mo","W",
    "Mn","Fe","Co","Ni","Cu",
    "Ru","Rh","Pd",
    "Os","Ir","Pt",
    "Sb","Bi"
]

X_SPECIES = {
    "O": -2,
    "F": -1,
    "Cl": -1,
    "Br": -1,
    "I": -1
}


# --------------------------------------------------
# Load OQMD compounds
# --------------------------------------------------

oqmd = pd.read_csv("oqmd_data.csv")

# first column contains formulas
existing = set(
    oqmd.iloc[:,0]
    .astype(str)
    .str.replace(" ", "")
)


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def oxidation_states(symbol):
    return [
        int(i)
        for i in Element(symbol).common_oxidation_states
        if i > 0
    ]


def shannon(symbol, oxidation, coord):

    try:
        return float(
            Species(symbol, oxidation).get_shannon_radius(coord)
        )
    except:
        return None


fallback_anion = {
    ("O",-2):1.40,
    ("F",-1):1.33,
    ("Cl",-1):1.81,
    ("Br",-1):1.96,
    ("I",-1):2.20
}


def anion_radius(symbol, oxidation):

    r = shannon(symbol, oxidation, "VI")

    if r is not None:
        return r

    return fallback_anion[(symbol, oxidation)]


def tolerance_factor(rA,rB,rX):

    return (rA+rX)/(math.sqrt(2)*(rB+rX))


def oct_factor(rB,rX):

    return rB/rX


def bartel_tau(rA,rB,rX,nA):

    ratio = rA/rB

    if ratio <= 1:
        return 999

    return (
        rX/rB
        - nA*(nA-ratio/math.log(ratio))
    )


# --------------------------------------------------
# Generate compounds
# --------------------------------------------------

novel = []

for A in A_ELEMENTS:

    for B in B_ELEMENTS:

        if A == B:
            continue

        for X,x_charge in X_SPECIES.items():

            for qA in oxidation_states(A):

                for qB in oxidation_states(B):

                    if qA + qB + 3*x_charge != 0:
                        continue

                    rA = shannon(A,qA,"XII")
                    rB = shannon(B,qB,"VI")
                    rX = anion_radius(X,x_charge)

                    if None in [rA,rB,rX]:
                        continue

                    if rA <= rB:
                        continue

                    t = tolerance_factor(rA,rB,rX)
                    mu = oct_factor(rB,rX)
                    tau = bartel_tau(rA,rB,rX,qA)

                    if not (0.80 <= t <= 1.00):
                        continue

                    if not (0.414 <= mu <= 0.732):
                        continue

                    if tau >= 4.18:
                        continue

                    formula = f"{A}{B}{X}3"

                    # Remove compounds already in OQMD
                    if formula in existing:
                        continue

                    novel.append(formula)


# --------------------------------------------------
# Output
# --------------------------------------------------

novel = sorted(set(novel))

df = pd.DataFrame({
    "compound": novel
})

print(df.head())

df.to_csv(
    "novel_perovskites.csv",
    index=False
)

  compound
0  CsPdBr3
1  CsPtCl3
2   KPtCl3
3    KPtF3
4   NaGeF3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 943.3 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.2 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=a7c61f9a738b2c8ce1a621d268d08cb36fe08458ca3606eed8d80ffba2dd00a3
  Stored in directory: /root/.cache/pip/wheels/54/f8/e6/ecfceb6af875ddc5096bb38117